In [ ]:
import os
import numpy as np
from glob import glob
import tifffile as tiff
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, precision_recall_curve, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import pandas as pd
import time
import lmdb
import pickle
from tqdm import tqdm
from torch.utils.data import Subset
import random
import gc
import zlib

# --- LMDB Patch Writer ---
def write_lmdb(images_dir, masks_dir, lmdb_path, patch_size=256, overlap=128):
    images = sorted(glob(os.path.join(images_dir, "*.tif")))
    map_size = 1024**4  # 1TB
    env = lmdb.open(lmdb_path, map_size=map_size)
    idx = 0
    with env.begin(write=True) as txn:
        for img_path in tqdm(images, desc=f"Writing {lmdb_path}"):
            img_name = os.path.basename(img_path)
            mask_name = f"mask_{img_name}"
            mask_path = os.path.join(masks_dir, mask_name)
            if not os.path.exists(mask_path):
                print(f"Mask not found for {img_name}, skipping.")
                continue
            img = tiff.imread(img_path)
            msk = tiff.imread(mask_path)
            height, width = img.shape[:2]
            for y in range(0, height - patch_size + 1, patch_size - overlap):
                for x in range(0, width - patch_size + 1, patch_size - overlap):
                    patch = img[y:y+patch_size, x:x+patch_size, :]
                    patch_mask = msk[y:y+patch_size, x:x+patch_size]
                    
                    # Robust NaN/Inf handling
                    if np.isnan(patch).any() or np.isinf(patch).any():
                        print(f"NaN or Inf found in patch {idx} from {img_name}")
                        patch = np.nan_to_num(patch, nan=0.0, posinf=1.0, neginf=0.0)
                    
                    patch = patch.astype(np.float32)

                    # Handle constant patches
                    if np.ptp(patch) > 1e-8:
                        
                        patch -= patch.min()
                        max_val = patch.max()
                        if max_val > 1e-8:
                            patch /= max_val
                        else:
                            patch[:] = 0
                    else:
                        patch[:] = 0
                    
                    patch = (patch * 255).astype(np.uint8)
                    txn.put(f"patch_{idx}".encode(), zlib.compress(patch.tobytes()))
                    txn.put(f"mask_{idx}".encode(), patch_mask.tobytes())
                    txn.put(f"shape_{idx}".encode(), pickle.dumps((patch.shape, patch_mask.shape)))
                    idx += 1
        txn.put(b'length', pickle.dumps(idx))
    env.close()
    print(f"LMDB created: {lmdb_path}, total patches: {idx}")

# --- LMDB Dataset ---
class LMDBPatchDataset(Dataset):
    def __init__(self, lmdb_path, patch_channels=3, patch_size=256):
        self.env = lmdb.open(lmdb_path, readonly=True, lock=False, readahead=False)
        with self.env.begin() as txn:
            self.length = pickle.loads(txn.get(b'length'))
        self.patch_channels = patch_channels
        self.patch_size = patch_size

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        with self.env.begin() as txn:
            patch_shape, mask_shape = pickle.loads(txn.get(f"shape_{idx}".encode()))
            patch = np.frombuffer(zlib.decompress(txn.get(f"patch_{idx}".encode())), dtype=np.uint8).reshape(patch_shape)
            patch = patch.astype(np.float32) / 255.0
            mask = np.frombuffer(txn.get(f"mask_{idx}".encode()), dtype=np.uint8).reshape(mask_shape)
        
        # Convert to tensors
        patch = torch.from_numpy(patch.transpose(2,0,1).copy()).float()
        mask = torch.from_numpy(mask.copy()).long()
        
        # Additional check for NaNs/Infs
        if torch.isnan(patch).any() or torch.isinf(patch).any():
            print(f"Found NaN value after reading batch {idx}")
            patch = torch.nan_to_num(patch, nan=0.0, posinf=1.0, neginf=0.0)
            
        return patch, mask

# --- Config ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

train_img_dir = "/home/btech1/isro/dataset/train1/images"
train_mask_dir = "/home/btech1/isro/dataset/train1/masks"
val_img_dir = "/home/btech1/isro/dataset/val1/images"
val_mask_dir = "/home/btech1/isro/dataset/val1/masks"
train_lmdb_path = "/home/btech1/isro/dataset/train1/patches.lmdb"
val_lmdb_path = "/home/btech1/isro/dataset/val1/patches.lmdb"
patch_size = 256
overlap = 0
batch_size = 6
num_classes = 3
classes = ["background", "cloud", "cloud_shadow"]
max_epochs = 3
early_stop_patience = 10
reduce_lr_patience = 5
min_lr = 1e-6
output_dir = "training_output"
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory: {os.path.abspath(output_dir)}")

# --- Check for LMDB ---
def lmdb_exists(lmdb_path):
    return os.path.exists(lmdb_path) and os.path.isfile(lmdb_path)

if not lmdb_exists(train_lmdb_path):
    print("Creating training LMDB...")
    write_lmdb(train_img_dir, train_mask_dir, train_lmdb_path, patch_size, overlap)
else:
    print("Training LMDB found.")

if not lmdb_exists(val_lmdb_path):
    print("Creating validation LMDB...")
    write_lmdb(val_img_dir, val_mask_dir, val_lmdb_path, patch_size, overlap)
else:
    print("Validation LMDB found.")

# --- DataLoaders ---
train_dataset = LMDBPatchDataset(train_lmdb_path, patch_channels=3, patch_size=patch_size)
val_dataset_full = LMDBPatchDataset(val_lmdb_path, patch_channels=3, patch_size=patch_size)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=8, pin_memory=True, persistent_workers=True)
print(f"Train dataset: {len(train_dataset)} patches")
print(f"Validation dataset: {len(val_dataset_full)} patches")
print(f"Batch size: {batch_size}")

# --- Model ---
class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, features=[64,128,256,512]):
        super(UNet, self).__init__()
        self.downs = nn.ModuleList()
        prev_channels = in_channels
        for feature in features:
            self.downs.append(nn.Sequential(
                nn.Conv2d(prev_channels, feature, 3, padding=1), 
                nn.InstanceNorm2d(feature, affine=True),
                nn.ReLU(),
                nn.Conv2d(feature, feature, 3, padding=1), 
                nn.InstanceNorm2d(feature, affine=True),
                nn.ReLU()))
            prev_channels = feature
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.bottleneck = nn.Sequential(
            nn.Conv2d(prev_channels, prev_channels*2, 3, padding=1), 
            nn.InstanceNorm2d(prev_channels*2, affine=True),
            nn.ReLU(),
            nn.Conv2d(prev_channels*2, prev_channels*2, 3, padding=1), 
            nn.InstanceNorm2d(prev_channels*2, affine=True),
            nn.ReLU())
        bottleneck_channels = prev_channels*2
        self.ups = nn.ModuleList()
        self.up_convs = nn.ModuleList()
        decoder_in_channels = bottleneck_channels
        for feature in reversed(features):
            self.ups.append(nn.ConvTranspose2d(decoder_in_channels, feature, kernel_size=2, stride=2))
            self.up_convs.append(nn.Sequential(
                nn.Conv2d(feature*2, feature, 3, padding=1), 
                nn.InstanceNorm2d(feature, affine=True),
                nn.ReLU(),
                nn.Conv2d(feature, feature, 3, padding=1), 
                nn.InstanceNorm2d(feature, affine=True),
                nn.ReLU()))
            decoder_in_channels = feature
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []
        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]
        for i in range(len(self.ups)):
            x = self.ups[i](x)
            skip = skip_connections[i]
            if x.shape != skip.shape:
                x = nn.functional.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
            x = torch.cat((skip, x), dim=1)
            x = self.up_convs[i](x)
        return self.final_conv(x)

model = UNet(in_channels=3, out_channels=3).to(device)

# Weight classes to handle imbalance
class_weights = torch.tensor([0.1, 1.0, 1.0]).float().to(device)  # Adjust based on your class distribution
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=reduce_lr_patience, min_lr=min_lr)

# --- Metrics Calculation ---
def compute_metrics(y_true, y_pred, num_classes=3):
    y_pred_flat = y_pred.flatten()
    y_true_flat = y_true.flatten()
    cm = confusion_matrix(y_true_flat, y_pred_flat, labels=list(range(num_classes)))
    total = cm.sum()
    TP = np.diag(cm)
    FP = cm.sum(axis=0) - TP
    FN = cm.sum(axis=1) - TP
    TN = total - (TP + FP + FN)
    
    # Overall accuracy
    overall_acc = TP.sum() / total if total else 0.0
    
    # Class accuracy
    class_acc = (TP + TN) / total
    
    # Precision, recall, F1
    precision = TP / (TP + FP + 1e-10)
    recall = TP / (TP + FN + 1e-10)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-10)
    
    # IoU
    iou = TP / (TP + FP + FN + 1e-10)
    
    # Macro averages
    macro_precision = np.mean(precision)
    macro_recall = np.mean(recall)
    macro_f1 = np.mean(f1)
    macro_iou = np.mean(iou)
    
    return {
        "overall_acc": overall_acc,
        "class_acc": class_acc.tolist(),
        "precision": precision.tolist(),
        "recall": recall.tolist(),
        "f1": f1.tolist(),
        "iou": iou.tolist(),
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "macro_iou": macro_iou
    }

# --- Training Loop ---
metrics_log = []
train_losses, val_losses, lrs = [], [], []
best_val_iou = 0
epochs_no_improve = 0

for epoch in range(1, max_epochs+1):
    print(f"\n=== Epoch {epoch}/{max_epochs} ===")
    epoch_start_time = time.time()
    model.train()
    running_train_loss = 0.0
    
    # Progress bar
    pbar = tqdm(total=len(train_loader), desc=f"Epoch {epoch}/{max_epochs}")

    train_all_preds = []
    train_all_trues = []

    for batch_idx, (images, masks) in enumerate(train_loader):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        unique_vals = torch.unique(masks)

        if not torch.all((unique_vals >= 0) & (unique_vals < num_classes)):
            print(f"Invalid mask values: {unique_vals}")

        # Skip batches with NaN/inf
        if torch.isnan(images).any() or torch.isinf(images).any():
            print(f"Skipping batch {batch_idx} due to NaN/Inf in input")
            pbar.update(1)
            continue
            
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)

        if torch.isnan(outputs).any() or torch.isinf(outputs).any():
            print(f"NaN/Inf in model output at batch {batch_idx}")
            print(f"NaN/Inf detected in model output! Batch shape: {outputs.shape}")
            print(f"Batch min: {outputs.min().item()}, max: {outputs.max().item()}, mean: {outputs.mean().item()}")
            print(f"Input images min: {images.min().item()}, max: {images.max().item()}, mean: {images.mean().item()}")

        loss.backward()

        # Skip NaN loss batches
        if torch.isnan(loss):
            print(f"Skipping batch {batch_idx} due to NaN loss")
            pbar.update(1)
            continue
            
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        # Calculate metrics
        with torch.no_grad():
            preds = outputs.argmax(dim=1).cpu().numpy()
            trues = masks.cpu().numpy()
            train_all_preds.append(preds)
            train_all_trues.append(trues)
            batch_metrics = compute_metrics(trues, preds, num_classes=3)
            
        running_train_loss += loss.item() * images.size(0)
        
        # Update progress bar
        # ...inside the pbar.set_postfix({...}) call...
        pbar.set_postfix({
            'Loss': f"{loss.item():.4f}", 
            'Accuracy': f"{np.mean(batch_metrics['class_acc']):.4f}",
            'IoU': f"{np.mean(batch_metrics['iou']):.3f}",
            'Precision': f"{batch_metrics['macro_precision']:.3f}",
            'Recall': f"{batch_metrics['macro_recall']:.3f}",
            'F1_Score': f"{batch_metrics['macro_f1']:.3f}",
            'LR': f"{optimizer.param_groups[0]['lr']:.6f}"
        })
        pbar.update(1)
    
    train_all_preds = np.concatenate(train_all_preds, axis=0)
    train_all_trues = np.concatenate(train_all_trues, axis=0)
    train_metrics = compute_metrics(train_all_trues, train_all_preds, num_classes=3)

    pbar.close()
    avg_train_loss = running_train_loss / len(train_loader.dataset)
    train_losses.append(avg_train_loss)
    epoch_time = time.time() - epoch_start_time
    print(f"Average Training Loss: {avg_train_loss:.4f}")
    print(f"Epoch {epoch} completed in {epoch_time:.1f} seconds.")

    # --- Validation ---
    # Use 1/5th of validation set
    val_len = len(val_dataset_full)
    val_indices = random.sample(range(val_len), val_len // 5)
    val_dataset = Subset(val_dataset_full, val_indices)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=8, pin_memory=True)

    print(f"Evaluating on {len(val_dataset)} validation patches...")
    model.eval()
    running_val_loss = 0.0
    all_preds = []
    all_trues = []
    
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)
            
            outputs = model(images)
            loss = criterion(outputs, masks)
            
            # Skip NaN batches
            if not torch.isnan(loss):
                running_val_loss += loss.item() * images.size(0)
                preds = outputs.argmax(dim=1).cpu().numpy()
                all_preds.append(preds)
                all_trues.append(masks.cpu().numpy())
    
    # Handle case where all batches were skipped
    if len(all_preds) == 0:
        print("WARNING: All validation batches skipped due to NaN loss!")
        avg_val_loss = float('nan')
        metrics = {
            "overall_acc": 0,
            "class_acc": [0, 0, 0],
            "precision": [0, 0, 0],
            "recall": [0, 0, 0],
            "f1": [0, 0, 0],
            "iou": [0, 0, 0],
            "macro_precision": 0,
            "macro_recall": 0,
            "macro_f1": 0,
            "macro_iou": 0
        }
    else:
        all_preds = np.concatenate(all_preds, axis=0)
        all_trues = np.concatenate(all_trues, axis=0)
        avg_val_loss = running_val_loss / len(val_loader.dataset)
        metrics = compute_metrics(all_trues, all_preds, num_classes=3)
    
    val_losses.append(avg_val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    lrs.append(current_lr)
    
    # Log metrics
    epoch_log = {
        "epoch": epoch,
        "learning_rate": current_lr,
        "train_loss": avg_train_loss,
        "train_accuracy": train_metrics["overall_acc"],
        "train_accuracy_background": train_metrics["class_acc"][0],
        "train_accuracy_cloud": train_metrics["class_acc"][1],
        "train_accuracy_shadow": train_metrics["class_acc"][2],
        "train_iou": train_metrics["macro_iou"],
        "train_iou_background": train_metrics["iou"][0],
        "train_iou_cloud": train_metrics["iou"][1],
        "train_iou_shadow": train_metrics["iou"][2],
        "train_precision": train_metrics["macro_precision"],
        "train_precision_background": train_metrics["precision"][0],
        "train_precision_cloud": train_metrics["precision"][1],
        "train_precision_shadow": train_metrics["precision"][2],
        "train_recall": train_metrics["macro_recall"],
        "train_recall_background": train_metrics["recall"][0],
        "train_recall_cloud": train_metrics["recall"][1],
        "train_recall_shadow": train_metrics["recall"][2],
        "train_f1-score": train_metrics["macro_f1"],
        "train_f1-score_background": train_metrics["f1"][0],
        "train_f1-score_cloud": train_metrics["f1"][1],
        "train_f1-score_shadow": train_metrics["f1"][2],
        "val_loss": avg_val_loss,
        "val_accuracy": metrics["overall_acc"],
        "val_accuracy_background": metrics["class_acc"][0],
        "val_accuracy_cloud": metrics["class_acc"][1],
        "val_accuracy_shadow": metrics["class_acc"][2],
        "val_iou": metrics["macro_iou"],
        "val_iou_background": metrics["iou"][0],
        "val_iou_cloud": metrics["iou"][1],
        "val_iou_shadow": metrics["iou"][2],
        "val_precision": metrics["macro_precision"],
        "val_precision_background": metrics["precision"][0],
        "val_precision_cloud": metrics["precision"][1],
        "val_precision_shadow": metrics["precision"][2],
        "val_recall": metrics["macro_recall"],
        "val_recall_background": metrics["recall"][0],
        "val_recall_cloud": metrics["recall"][1],
        "val_recall_shadow": metrics["recall"][2],
        "val_f1-score": metrics["macro_f1"],
        "val_f1-score_background": metrics["f1"][0],
        "val_f1-score_cloud": metrics["f1"][1],
        "val_f1-score_shadow": metrics["f1"][2],
    }
    metrics_log.append(epoch_log)
    
    # Save metrics immediately after each epoch
    df = pd.DataFrame(metrics_log)
    df.to_csv(os.path.join(output_dir, "metrics_log.csv"), index=False)
    print("Saved metrics_log.csv")

    print(f"Epoch {epoch}: LR={current_lr:.6f}, Train Loss={avg_train_loss:.4f}, "
      f"Accuracy={train_metrics['overall_acc']:.4f}, Cloud IoU={train_metrics['macro_iou']:.3f}, "
      f"Precision={train_metrics['macro_precision']:.3f}, Recall={train_metrics['macro_recall']:.3f}, F1_Score={train_metrics['macro_f1']:.3f}, "
      f"Val_Loss={avg_val_loss:.4f}, Val_Accuracy={metrics['overall_acc']:.3f}, "
      f"Val_IoU={metrics['macro_iou']:.3f}, Val_Precision={metrics['macro_precision']:.3f}, "
      f"Val_Recall={metrics['macro_recall']:.3f}, Val_F1={metrics['macro_f1']:.3f}")

    # Update scheduler and check for improvement
    scheduler.step(metrics["macro_iou"])
    if metrics["macro_iou"] > best_val_iou:
        best_val_iou = metrics["macro_iou"]
        epochs_no_improve = 0
        torch.save(model.state_dict(), os.path.join(output_dir, f"best_model_epoch_{epoch}.pth"))
        print(f"Saved best model checkpoint at epoch {epoch}.")
    else:
        epochs_no_improve += 1
        print(f"No improvement for {epochs_no_improve} epochs.")
        if epochs_no_improve >= early_stop_patience:
            print("Early stopping triggered.")
            break

# --- Save Final Model ---
torch.save(model.state_dict(), os.path.join(output_dir, "final_model.pth"))
print("Saved final model weights.")

# --- Plotting Functions ---
def plot_metrics(metric_name, classes, df, output_dir):
    """Plot overall and per-class metrics for both train and val"""
    plt.figure(figsize=(12, 8))
    
    # Plot train and val overall metrics
    if f"train_{metric_name}" in df.columns:
        plt.plot(df['epoch'], df[f"train_{metric_name}"], label=f'Train {metric_name}', linewidth=2, linestyle='-')
    if f"val_{metric_name}" in df.columns:
        plt.plot(df['epoch'], df[f"val_{metric_name}"], label=f'Val {metric_name}', linewidth=2, linestyle='--')
    
    # Per-class metrics
    for i, cls in enumerate(classes):
        if f"train_{metric_name}_class{i}" in df.columns:
            plt.plot(df['epoch'], df[f"train_{metric_name}_class{i}"], label=f'Train {cls} {metric_name}', alpha=0.5)
        if f"val_{metric_name}_class{i}" in df.columns:
            plt.plot(df['epoch'], df[f"val_{metric_name}_class{i}"], label=f'Val {cls} {metric_name}', alpha=0.5, linestyle='--')
    
    plt.title(f"{metric_name} vs Epoch")
    plt.xlabel("Epoch")
    plt.ylabel(metric_name)
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, f"{metric_name.lower()}_vs_epoch.png"))
    plt.close()
    print(f"Saved {metric_name.lower()}_vs_epoch.png")

def plot_all_metrics(df, classes, output_dir):
    """Generate all required metric plots"""
    # Accuracy
    plot_metrics("accuracy", classes, df, output_dir)
    # IoU
    plot_metrics("iou", classes, df, output_dir)
    # Precision
    plot_metrics("precision", classes, df, output_dir)
    # Recall
    plot_metrics("recall", classes, df, output_dir)
    # F1-Score
    plot_metrics("f1-score", classes, df, output_dir)
    # Loss
    plt.figure(figsize=(12, 8))
    plt.plot(df['epoch'], df['train_loss'], label='Training Loss')
    plt.plot(df['epoch'], df['val_loss'], label='Validation Loss')
    plt.title("Loss vs Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, "loss_vs_epoch.png"))
    plt.close()
    print("Saved loss_vs_epoch.png")
    # Learning rate
    plt.figure(figsize=(12, 8))
    plt.plot(df['epoch'], df['learning_rate'], label='Learning Rate')
    plt.title("Learning Rate vs Epoch")
    plt.xlabel("Epoch")
    plt.ylabel("Learning Rate")
    plt.yscale('log')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, "lr_vs_epoch.png"))
    plt.close()
    print("Saved lr_vs_epoch.png")

# --- Generate All Plots ---
print("\nGenerating plots...")
df = pd.read_csv(os.path.join(output_dir, "metrics_log.csv"))
if not df.empty:
    plot_all_metrics(df, classes, output_dir)
else:
    print("No metrics to plot!")

# --- Additional Analysis Plots ---
try:
    # Precision-Recall Curve
    print("Generating precision-recall curve...")
    model.eval()
    N = min(300, len(val_dataset_full))  # Use up to 200 patches for speed
    val_indices = random.sample(range(len(val_dataset_full)), N)
    all_true = [[] for _ in range(len(classes))]
    all_score = [[] for _ in range(len(classes))]

    with torch.no_grad():
        for idx in val_indices:
            img, true_mask = val_dataset_full[idx]
            img = img.unsqueeze(0).to(device)
            output = model(img)
            probas = torch.softmax(output, dim=1).cpu().numpy()[0]  # shape: (C, H, W)
            true_mask = true_mask.numpy()
            for i in range(len(classes)):
                all_true[i].append((true_mask == i).astype(int).flatten())
                all_score[i].append(probas[i].flatten())

    # Concatenate all patches for each class
    plt.figure(figsize=(10, 8))
    for i, cls in enumerate(classes):
        y_true_cls = np.concatenate(all_true[i])
        y_score_cls = np.concatenate(all_score[i])
        if np.sum(y_true_cls) == 0:
            print(f"Class '{cls}' not present in any selected patches, skipping PR curve.")
            continue
        prec, rec, _ = precision_recall_curve(y_true_cls, y_score_cls)
        plt.plot(rec, prec, label=f"Class {cls}", linewidth=2)

    plt.title("Precision-Recall Curves (Aggregated)")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, "precision_recall_curves.png"))
    plt.close()
    print("Saved precision_recall_curves.png")
    
    # Confusion Matrix
    print("Generating confusion matrix...")
    val_indices = random.sample(range(len(val_dataset_full)), min(1000, len(val_dataset_full)))
    val_subset = Subset(val_dataset_full, val_indices)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)
    
    all_preds = []
    all_trues = []
    model.eval()
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.append(preds)
            all_trues.append(masks.cpu().numpy())
    
    all_preds = np.concatenate(all_preds, axis=0).flatten()
    all_trues = np.concatenate(all_trues, axis=0).flatten()
    
    cm = confusion_matrix(all_trues, all_preds, labels=[0,1,2])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    fig, ax = plt.subplots(figsize=(10,8))
    disp.plot(ax=ax, cmap='Blues', values_format='d')
    plt.title("Confusion Matrix")
    plt.savefig(os.path.join(output_dir, "confusion_matrix.png"))
    plt.close()
    print("Saved confusion_matrix.png")
    
    # Segmentation Heatmap
    print("Generating segmentation heatmap...")
    model.eval()
    with torch.no_grad():
        for idx in range(len(val_dataset_full)):
            _, true_mask = val_dataset_full[idx]
            unique = np.unique(true_mask.numpy())
            if len(unique) > 2:
                print(f"Patch {idx} has classes: {unique}")
                break
        img, true_mask = val_dataset_full[idx]
        output = model(img.unsqueeze(0).to(device))
        pred_mask = output.argmax(dim=1).squeeze(0).cpu().numpy()
    
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.imshow(true_mask, cmap='viridis')
    plt.title("True Mask")
    plt.colorbar(ticks=[0,1,2], label='Class')
    
    plt.subplot(1, 2, 2)
    plt.imshow(pred_mask, cmap='viridis')
    plt.title("Predicted Mask")
    plt.colorbar(ticks=[0,1,2], label='Class')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "segmentation_heatmap.png"))
    plt.close()
    print("Saved segmentation_heatmap.png")
    
except Exception as e:
    print(f"Error generating additional plots: {str(e)}")

print("\nTraining complete! All outputs saved to:", os.path.abspath(output_dir))

Using device: cuda
Output directory: /home/btech1/isro/code/training_output
Creating training LMDB...


Writing /home/btech1/isro/dataset/train1/patches.lmdb: 100%|██████████| 1/1 [00:03<00:00,  3.37s/it]


LMDB created: /home/btech1/isro/dataset/train1/patches.lmdb, total patches: 1085
Creating validation LMDB...


Writing /home/btech1/isro/dataset/val1/patches.lmdb: 100%|██████████| 1/1 [00:03<00:00,  3.52s/it]


LMDB created: /home/btech1/isro/dataset/val1/patches.lmdb, total patches: 1088
Train dataset: 1085 patches
Validation dataset: 1088 patches
Batch size: 6

=== Epoch 1/3 ===


...

Epoch 3/3: 100%|██████████| 181/181 [01:13<00:00,  2.45it/s, Loss=0.4831, Accuracy=0.7676, IoU=0.468, Precision=0.612, Recall=0.800, F1_Score=0.637, LR=0.000100]

Average Training Loss: 0.4446
Epoch 3 completed in 73.7 seconds.
Evaluating on 217 validation patches...


Saved metrics_log.csv
Epoch 3: LR=0.000100, Train Loss=0.4446, Accuracy=0.7333, Cloud IoU=0.570, Precision=0.706, Recall=0.835, F1_Score=0.718, Val_Loss=0.4337, Val_Accuracy=0.671, Val_IoU=0.494, Val_Precision=0.633, Val_Recall=0.850, Val_F1=0.642
Saved best model checkpoint at epoch 3.
Saved final model weights.

Generating plots...
Saved accuracy_vs_epoch.png
Saved iou_vs_epoch.png
Saved precision_vs_epoch.png
Saved recall_vs_epoch.png
Saved f1-score_vs_epoch.png
Saved loss_vs_epoch.png
Saved lr_vs_epoch.png
Generating precision-recall curve...


/tmp/ipykernel_2238847/933339749.py:590: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(os.path.join(output_dir, "precision_recall_curves.png"))


Saved precision_recall_curves.png
Generating confusion matrix...
Saved confusion_matrix.png
Generating segmentation heatmap...
Saved segmentation_heatmap.png

Training complete! All outputs saved to: /home/btech1/isro/code/training_output
